In [1]:
!pip -q install -U "lm-eval[hf]" transformers accelerate datasets sentencepiece

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


In [2]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

GPU: Tesla T4
CUDA available: True


In [6]:
!pip -q install langdetect

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [8]:
import json
import subprocess

SAMPLES = {
    "gsm8k": list(range(200)),
    "ifeval": list(range(200)),
}

BASE_CMD = [
    "lm-eval",
    "--model", "hf",
    "--model_args",
    "pretrained=allenai/OLMo-2-0425-1B,dtype=float16",
    "--tasks", "gsm8k,ifeval",
    "--num_fewshot", "0",
    "--samples", json.dumps(SAMPLES),
    "--batch_size", "1",
    "--device", "cuda:0",
    "--gen_kwargs", "max_gen_toks=512,do_sample=False",
    "--output_path", "./results/base",
    "--log_samples",
]

subprocess.run(BASE_CMD, check=True)


CompletedProcess(args=['lm-eval', '--model', 'hf', '--model_args', 'pretrained=allenai/OLMo-2-0425-1B,dtype=float16', '--tasks', 'gsm8k,ifeval', '--num_fewshot', '0', '--samples', '{"gsm8k": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183

In [13]:
import json
from pathlib import Path

def read_scores(folder):
    files = sorted(
        Path(folder).rglob("results_*.json"),
        key=lambda p: p.stat().st_mtime
    )
    if not files:
        raise FileNotFoundError(folder)

    with open(files[-1], "r") as f:
        data = json.load(f)

    for task in ["gsm8k", "ifeval"]:
        print(f"{task.upper()}")
        for metric, value in data["results"][task].items():
            if isinstance(value, (int, float)) and "stderr" not in metric:
                print(f"  {metric}: {value:.4f}")

In [14]:
read_scores("./results/base")

GSM8K
  sample_len: 200.0000
  exact_match,strict-match: 0.0000
  exact_match,flexible-extract: 0.0400
IFEVAL
  sample_len: 200.0000
  prompt_level_strict_acc,none: 0.0800
  inst_level_strict_acc,none: 0.1855
  prompt_level_loose_acc,none: 0.0950
  inst_level_loose_acc,none: 0.1950


In [15]:
SFT_CMD = [
    "lm-eval",
    "--model", "hf",
    "--model_args",
    "pretrained=allenai/OLMo-2-0425-1B-SFT,dtype=float16",
    "--tasks", "gsm8k,ifeval",
    "--num_fewshot", "0",
    "--samples", json.dumps(SAMPLES),
    "--batch_size", "1",
    "--device", "cuda:0",
    "--gen_kwargs", "max_gen_toks=512,do_sample=False",
    "--apply_chat_template",
    "--output_path", "./results/sft",
    "--log_samples",
]

subprocess.run(SFT_CMD, check=True)

CompletedProcess(args=['lm-eval', '--model', 'hf', '--model_args', 'pretrained=allenai/OLMo-2-0425-1B-SFT,dtype=float16', '--tasks', 'gsm8k,ifeval', '--num_fewshot', '0', '--samples', '{"gsm8k": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182,

In [17]:
read_scores("./results/sft")

GSM8K
  sample_len: 200.0000
  exact_match,strict-match: 0.0000
  exact_match,flexible-extract: 0.4750
IFEVAL
  sample_len: 200.0000
  prompt_level_strict_acc,none: 0.3800
  inst_level_strict_acc,none: 0.5189
  prompt_level_loose_acc,none: 0.4150
  inst_level_loose_acc,none: 0.5566


In [18]:
DPO_CMD = [
    "lm-eval",
    "--model", "hf",
    "--model_args",
    "pretrained=allenai/OLMo-2-0425-1B-DPO,dtype=float16",
    "--tasks", "gsm8k,ifeval",
    "--num_fewshot", "0",
    "--samples", json.dumps(SAMPLES),
    "--batch_size", "1",
    "--device", "cuda:0",
    "--gen_kwargs", "max_gen_toks=512,do_sample=False",
    "--apply_chat_template",
    "--output_path", "./results/dpo",
    "--log_samples",
]

subprocess.run(DPO_CMD, check=True)

CompletedProcess(args=['lm-eval', '--model', 'hf', '--model_args', 'pretrained=allenai/OLMo-2-0425-1B-DPO,dtype=float16', '--tasks', 'gsm8k,ifeval', '--num_fewshot', '0', '--samples', '{"gsm8k": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182,

In [19]:
read_scores("./results/dpo")

GSM8K
  sample_len: 200.0000
  exact_match,strict-match: 0.0000
  exact_match,flexible-extract: 0.4700
IFEVAL
  sample_len: 200.0000
  prompt_level_strict_acc,none: 0.5350
  inst_level_strict_acc,none: 0.6541
  prompt_level_loose_acc,none: 0.5750
  inst_level_loose_acc,none: 0.6887


In [20]:
INSTRUCT_CMD = [
    "lm-eval",
    "--model", "hf",
    "--model_args",
    "pretrained=allenai/OLMo-2-0425-1B-Instruct,dtype=float16",
    "--tasks", "gsm8k,ifeval",
    "--num_fewshot", "0",
    "--samples", json.dumps(SAMPLES),
    "--batch_size", "1",
    "--device", "cuda:0",
    "--gen_kwargs", "max_gen_toks=512,do_sample=False",
    "--apply_chat_template",
    "--output_path", "./results/instruct",
    "--log_samples",
]

subprocess.run(INSTRUCT_CMD, check=True)

CompletedProcess(args=['lm-eval', '--model', 'hf', '--model_args', 'pretrained=allenai/OLMo-2-0425-1B-Instruct,dtype=float16', '--tasks', 'gsm8k,ifeval', '--num_fewshot', '0', '--samples', '{"gsm8k": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181,

In [21]:
read_scores("./results/instruct")

GSM8K
  sample_len: 200.0000
  exact_match,strict-match: 0.0000
  exact_match,flexible-extract: 0.6800
IFEVAL
  sample_len: 200.0000
  prompt_level_strict_acc,none: 0.5550
  inst_level_strict_acc,none: 0.6792
  prompt_level_loose_acc,none: 0.6150
  inst_level_loose_acc,none: 0.7264


In [22]:
import json
from pathlib import Path

for model_name in ["base", "sft", "dpo", "instruct"]:
    files = sorted(
        Path(f"./results/{model_name}").rglob("samples_*.jsonl")
    )
    print(f"\n{model_name.upper()}")
    for file in files:
        print(" ", file.name)

    if files:
        with open(files[0], encoding="utf-8") as f:
            sample = json.loads(next(f))
        print("Sample fields:", list(sample.keys()))
        for key in ["doc_id", "doc", "target", "resps", "filtered_resps"]:
            if key in sample:
                print(f"{key}: {str(sample[key])[:400]}")


BASE
  samples_gsm8k_2026-11-12T02-48-33.064395.jsonl
  samples_ifeval_2026-11-12T02-48-33.064395.jsonl
Sample fields: ['doc_id', 'doc', 'target', 'arguments', 'resps', 'filtered_resps', 'filter', 'metrics', 'doc_hash', 'prompt_hash', 'target_hash', 'exact_match']
doc_id: 0
doc: {'question': "Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", 'answer': 'Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 eve
target: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
resps: [[' 16 eggs x $2 = $32\n']]
filtered_resps: ['[invalid]']

SFT
  samples_gsm8k_2026-11-12T03-24-02.412420.jsonl
  samples_ifeval_2026-11-12T03-24-02.412420.jsonl
Sample fiel

In [23]:
from pathlib import Path
import json

MODELS = ["base", "sft", "dpo", "instruct"]

def load_samples(model, task):
    files = sorted(
        Path(f"./results/{model}").rglob(f"samples_{task}_*.jsonl")
    )
    if not files:
        raise FileNotFoundError(f"No samples for {model}/{task}")

    with open(files[-1], encoding="utf-8") as f:
        return [json.loads(line) for line in f]

# Inspect the same IFEval question across all checkpoints
for model in MODELS:
    samples = load_samples(model, "ifeval")
    sample = next(s for s in samples if s["doc_id"] == 0)

    print(f"\n{'=' * 20} {model.upper()} {'=' * 20}")
    if model == "base":
        print("PROMPT:", sample["doc"])

    print("RESPONSE:", str(sample["resps"][0][0])[:1200])
    print("METRICS:", sample.get("metrics"))


==================== BASE ====================
PROMPT: {'key': 1000, 'prompt': 'Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/Raymond_III,_Count_of_Tripoli". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*.', 'instruction_id_list': ['punctuation:no_comma', 'detectable_format:number_highlighted_sections', 'length_constraints:number_words'], 'kwargs': [{'num_highlights': None, 'relation': None, 'num_words': None, 'num_placeholders': None, 'prompt_to_repeat': None, 'num_bullets': None, 'section_spliter': None, 'num_sections': None, 'capital_relation': None, 'capital_frequency': None, 'keywords': None, 'num_paragraphs': None, 'language': None, 'let_relation': None, 'letter': None, 'let_frequency': None, 'end_phrase': None, 'forbidden_words': None, 'keyword': None, 'frequency': None, 'num_sentences': None, 'postscr

In [24]:
for model in MODELS:
    samples = load_samples(model, "ifeval")
    sample = next(s for s in samples if s["doc_id"] == 0)

    response = sample["resps"][0][0]

    print(f"\n{'=' * 20} {model.upper()} {'=' * 20}")
    print("Word count:", len(response.split()))
    print("Comma count:", response.count(","))
    print("Response length:", len(response))

    print("\nEvaluation results:")
    for key, value in sample.items():
        if "acc" in key:
            print(f"  {key}: {value}")

    print("\nFull response:")
    print(response)


==================== BASE ====================
Word count: 284
Comma count: 26
Response length: 2059

Evaluation results:
  prompt_level_strict_acc: False
  inst_level_strict_acc: [False, True, True]
  prompt_level_loose_acc: False
  inst_level_loose_acc: [False, True, True]

Full response:
 3. Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/William,_Earl_of_Hertford". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*. 4. Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/William,_Earl_of_Hertford". Do not use any commas and highlight at least 3 sections that has titles in markdown format, for example *highlighted section part 1*, *highlighted section part 2*, *highlighted section part 3*. 5. Write a 300+ word summary of the wikipedia page "https://en.wikipedia.org/wiki/William,_Earl_of